In [0]:
%sql

WITH merchants_actuales AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY merchant_id
            ORDER BY TRY_TO_TIMESTAMP(updated_at) DESC
        ) AS rn
    FROM kipu_pay.bronze.merchants
),

por_pais AS (
    SELECT
        m.merchant_id,
        m.merchant_name,
        m.country,
        t.currency,
        SUM(
            CAST(REPLACE(t.amount, ',', '.') AS DECIMAL(15,2))
        ) AS tpv
    FROM merchants_actuales m
    JOIN kipu_pay.bronze.transactions t
        ON m.merchant_id = t.merchant_id
    WHERE m.rn = 1
      AND t.currency = 'USD'
      AND t.status = 'approved'
      AND COALESCE(
            TRY_TO_TIMESTAMP(t.created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(t.created_at, 'yyyy-MM-dd HH:mm:ss')
          ) >= TIMESTAMP '2026-10-01 00:00:00'
      AND COALESCE(
            TRY_TO_TIMESTAMP(t.created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(t.created_at, 'yyyy-MM-dd HH:mm:ss')
          ) < TIMESTAMP '2026-11-01 00:00:00'
    GROUP BY
        m.merchant_id,
        m.merchant_name,
        m.country,
        t.currency
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY tpv DESC
        ) AS posicion
    FROM por_pais
)

SELECT *
FROM ranking
WHERE posicion <= 10
ORDER BY country, posicion;

In [0]:
%sql

WITH merchants_actuales AS (
   SELECT
    *,
    CASE
        WHEN UPPER(TRIM(country)) IN ('AR', 'ARG', 'ARGENTINA') THEN 'AR'
        WHEN UPPER(TRIM(country)) IN ('CL', 'CHL', 'CHILE') THEN 'CL'
        WHEN UPPER(TRIM(country)) IN ('CO', 'COL', 'COLOMBIA') THEN 'CO'
        WHEN UPPER(TRIM(country)) IN ('MX', 'MEX', 'MEXICO', 'MÉXICO') THEN 'MX'
        ELSE UPPER(TRIM(country))
    END AS country_normalized,
    ROW_NUMBER() OVER (
        PARTITION BY merchant_id
        ORDER BY TRY_TO_TIMESTAMP(updated_at) DESC
    ) AS rn
FROM kipu_pay.bronze.merchants
),

transacciones_limpias AS (
    SELECT
        transaction_id,
        merchant_id,
        UPPER(TRIM(currency)) AS currency,
        LOWER(TRIM(status)) AS status,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha,
        TRY_CAST(REPLACE(amount, ',', '.') AS DECIMAL(18,2)) AS amount
    FROM kipu_pay.bronze.transactions
),

fx_limpias AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(units_per_usd AS DECIMAL(18,6)) AS units_per_usd,
        published_at
    FROM kipu_pay.bronze.fx_rates
),

tpv_comercios AS (
    SELECT
        m.merchant_id,
        m.merchant_name,
        m.country_normalized AS country,
        SUM(
            t.amount / NULLIF(f.units_per_usd, CAST(0 AS DECIMAL(18,6)))
        ) AS tpv_usd
    FROM transacciones_limpias t
    INNER JOIN merchants_actuales m
        ON t.merchant_id = m.merchant_id
       AND m.rn = 1
    INNER JOIN fx_limpias f
        ON t.currency = f.currency
       AND TO_DATE(t.fecha) = f.rate_date
    WHERE t.status = 'approved'
      AND t.fecha >= TIMESTAMP '2026-10-01 00:00:00'
      AND t.fecha < TIMESTAMP '2026-11-01 00:00:00'
      AND t.amount IS NOT NULL
      AND f.units_per_usd > 0
    GROUP BY
        m.merchant_id,
        m.merchant_name,
        m.country_normalized
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY tpv_usd DESC
        ) AS posicion
    FROM tpv_comercios
)

SELECT
    country,
    posicion,
    merchant_id,
    merchant_name,
    ROUND(tpv_usd, 2) AS tpv_usd
FROM ranking
WHERE posicion <= 10
ORDER BY country, posicion;

In [0]:
%sql
WITH transacciones AS (
    SELECT
        transaction_id,
        UPPER(TRIM(currency)) AS currency,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),
fx AS (
    SELECT DISTINCT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
)
SELECT
    TO_DATE(t.fecha) AS dia,
    t.currency,
    COUNT(*) AS transacciones_sin_fx
FROM transacciones t
WHERE t.fecha >= TIMESTAMP '2026-10-01 00:00:00'
  AND t.fecha <  TIMESTAMP '2026-11-01 00:00:00'
  AND t.fecha IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM fx f
      WHERE f.rate_date = TO_DATE(t.fecha)
        AND f.currency = t.currency
  )
GROUP BY TO_DATE(t.fecha), t.currency
ORDER BY dia, t.currency;

In [0]:
%sql
SELECT
    rate_date,
    currency,
    source,
    COUNT(*) AS registros
FROM kipu_pay.bronze.fx_rates
WHERE TRY_CAST(rate_date AS DATE) BETWEEN
    DATE '2026-10-01' AND DATE '2026-10-31'
GROUP BY rate_date, currency, source
ORDER BY rate_date, currency;

In [0]:
%sql
WITH transacciones AS (
    SELECT
        transaction_id,
        UPPER(TRIM(currency)) AS currency,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),
fx AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(units_per_usd AS DECIMAL(18,6)) AS units_per_usd
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
)
SELECT
    t.transaction_id,
    t.currency,
    TO_DATE(t.fecha) AS fecha_transaccion,
    MAX(f.rate_date) AS fecha_cotizacion
FROM transacciones t
LEFT JOIN fx f
    ON t.currency = f.currency
   AND f.rate_date <= TO_DATE(t.fecha)
WHERE t.fecha >= TIMESTAMP '2026-10-01 00:00:00'
  AND t.fecha < TIMESTAMP '2026-11-01 00:00:00'
  AND t.fecha IS NOT NULL
  AND t.currency <> 'USD'
GROUP BY
    t.transaction_id,
    t.currency,
    TO_DATE(t.fecha)
ORDER BY fecha_transaccion
LIMIT 30;

In [0]:
%sql
-- Mismo criterio de búsqueda, limitado al 3 y 4 de octubre
WITH transacciones AS (
    SELECT
        transaction_id,
        UPPER(TRIM(currency)) AS currency,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),
fx AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
)
SELECT
    t.transaction_id,
    t.currency,
    TO_DATE(t.fecha) AS fecha_transaccion,
    MAX(f.rate_date) AS fecha_cotizacion
FROM transacciones t
LEFT JOIN fx f
    ON t.currency = f.currency
   AND f.rate_date <= TO_DATE(t.fecha)
WHERE TO_DATE(t.fecha) IN (DATE '2026-10-03', DATE '2026-10-04')
  AND t.currency <> 'USD'
GROUP BY t.transaction_id, t.currency, TO_DATE(t.fecha)
LIMIT 20;

In [0]:
%sql
WITH transacciones AS (
    SELECT
        transaction_id,
        UPPER(TRIM(currency)) AS currency,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),
fx AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
    GROUP BY
        TRY_CAST(rate_date AS DATE),
        UPPER(TRIM(currency))
),
cotizaciones_asignadas AS (
    SELECT
        t.transaction_id,
        t.currency,
        TO_DATE(t.fecha) AS fecha_transaccion,
        f.rate_date AS fecha_cotizacion,
        ROW_NUMBER() OVER (
            PARTITION BY t.transaction_id
            ORDER BY f.rate_date DESC
        ) AS rn
    FROM transacciones t
    LEFT JOIN fx f
        ON t.currency = f.currency
       AND f.rate_date <= TO_DATE(t.fecha)
    WHERE t.fecha >= TIMESTAMP '2026-10-01 00:00:00'
      AND t.fecha < TIMESTAMP '2026-11-01 00:00:00'
      AND t.fecha IS NOT NULL
      AND t.currency <> 'USD'
)
SELECT
    COUNT(*) AS total_transacciones,
    SUM(CASE WHEN fecha_cotizacion IS NOT NULL THEN 1 ELSE 0 END) AS con_cotizacion,
    SUM(CASE WHEN fecha_cotizacion IS NULL THEN 1 ELSE 0 END) AS sin_cotizacion
FROM cotizaciones_asignadas
WHERE rn = 1;

In [0]:
%sql
WITH merchants_actuales AS (
    SELECT
        merchant_id,
        merchant_name,
        CASE
            WHEN UPPER(TRIM(country)) IN ('AR', 'ARG', 'ARGENTINA') THEN 'AR'
            WHEN UPPER(TRIM(country)) IN ('CL', 'CHL', 'CHILE') THEN 'CL'
            WHEN UPPER(TRIM(country)) IN ('CO', 'COL', 'COLOMBIA') THEN 'CO'
            WHEN UPPER(TRIM(country)) IN ('MX', 'MEX', 'MEXICO', 'MÉXICO') THEN 'MX'
            ELSE UPPER(TRIM(country))
        END AS country,
        ROW_NUMBER() OVER (
            PARTITION BY merchant_id
            ORDER BY TRY_TO_TIMESTAMP(updated_at) DESC
        ) AS rn
    FROM kipu_pay.bronze.merchants
),

transacciones AS (
    SELECT
        transaction_id,
        merchant_id,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(REPLACE(amount, ',', '.') AS DECIMAL(18,2)) AS amount,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),

fx AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(units_per_usd AS DECIMAL(18,6)) AS units_per_usd
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
),

transacciones_fx AS (
    SELECT
        t.transaction_id,
        t.merchant_id,
        t.currency,
        t.amount,
        CASE
            WHEN t.currency = 'USD' THEN t.amount
            ELSE t.amount / NULLIF(f.units_per_usd, 0)
        END AS amount_usd,
        ROW_NUMBER() OVER (
            PARTITION BY t.transaction_id
            ORDER BY f.rate_date DESC
        ) AS rn
    FROM transacciones t
    LEFT JOIN fx f
        ON t.currency = f.currency
       AND f.rate_date <= TO_DATE(t.fecha)
    WHERE t.fecha >= TIMESTAMP '2026-10-01 00:00:00'
      AND t.fecha < TIMESTAMP '2026-11-01 00:00:00'
      AND t.fecha IS NOT NULL
),

tpv_comercios AS (
    SELECT
        m.merchant_id,
        m.merchant_name,
        m.country,
        SUM(t.amount_usd) AS tpv_usd
    FROM transacciones_fx t
    INNER JOIN merchants_actuales m
        ON t.merchant_id = m.merchant_id
       AND m.rn = 1
    WHERE t.rn = 1
      AND t.amount_usd IS NOT NULL
    GROUP BY
        m.merchant_id,
        m.merchant_name,
        m.country
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY tpv_usd DESC
        ) AS posicion
    FROM tpv_comercios
)

SELECT
    country,
    posicion,
    merchant_id,
    merchant_name,
    ROUND(tpv_usd, 2) AS tpv_usd
FROM ranking
WHERE posicion <= 10
ORDER BY country, posicion;

In [0]:
%sql
WITH transacciones AS (
    SELECT
        transaction_id,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(REPLACE(amount, ',', '.') AS DECIMAL(18,2)) AS amount,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),
fx AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(units_per_usd AS DECIMAL(18,6)) AS units_per_usd
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
),
asignacion AS (
    SELECT
        t.*,
        f.units_per_usd,
        ROW_NUMBER() OVER (
            PARTITION BY t.transaction_id
            ORDER BY f.rate_date DESC
        ) AS rn
    FROM transacciones t
    LEFT JOIN fx f
        ON t.currency = f.currency
       AND f.rate_date <= TO_DATE(t.fecha)
    WHERE t.fecha >= TIMESTAMP '2026-10-01 00:00:00'
      AND t.fecha < TIMESTAMP '2026-11-01 00:00:00'
      AND t.fecha IS NOT NULL
)
SELECT
    currency,
    COUNT(*) AS transacciones,
    ROUND(SUM(
        CASE
            WHEN currency = 'USD' THEN amount
            ELSE amount / NULLIF(units_per_usd, 0)
        END
    ), 2) AS tpv_usd
FROM asignacion
WHERE rn = 1
GROUP BY currency
ORDER BY currency;

In [0]:
%sql
SELECT
    transaction_id,
    amount,
    created_at,
    merchant_id
FROM kipu_pay.bronze.transactions
WHERE LOWER(TRIM(status)) = 'approved'
  AND UPPER(TRIM(currency)) = 'USD'
  AND COALESCE(
      TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
      TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
      TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
      TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
  ) >= TIMESTAMP '2026-10-01 00:00:00'
  AND COALESCE(
      TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
      TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
      TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
      TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
  ) < TIMESTAMP '2026-11-01 00:00:00'
ORDER BY TRY_CAST(REPLACE(amount, ',', '.') AS DECIMAL(18,2)) DESC
LIMIT 15;

In [0]:
%sql
SELECT
    transaction_id,
    COUNT(*) AS repeticiones,
    MAX(amount) AS importe,
    MAX(currency) AS moneda
FROM kipu_pay.bronze.transactions
WHERE UPPER(TRIM(currency)) = 'USD'
  AND LOWER(TRIM(status)) = 'approved'
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY repeticiones DESC;

In [0]:
%sql
WITH transacciones_parseadas AS (
    SELECT
        transaction_id,
        merchant_id,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(REPLACE(amount, ',', '.') AS DECIMAL(18,2)) AS amount,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),

-- Octubre y deduplicación por transaction_id
transacciones AS (
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY transaction_id
                ORDER BY fecha DESC
            ) AS rn
        FROM transacciones_parseadas
        WHERE fecha >= TIMESTAMP '2026-10-01 00:00:00'
          AND fecha < TIMESTAMP '2026-11-01 00:00:00'
          AND amount IS NOT NULL
    )
    WHERE rn = 1
),

-- Último registro disponible de cada comercio
comercios AS (
    SELECT merchant_id, merchant_name,
        CASE
            WHEN UPPER(TRIM(country)) IN ('AR','ARG','ARGENTINA') THEN 'AR'
            WHEN UPPER(TRIM(country)) IN ('CL','CHL','CHILE') THEN 'CL'
            WHEN UPPER(TRIM(country)) IN ('CO','COL','COLOMBIA') THEN 'CO'
            WHEN UPPER(TRIM(country)) IN ('MX','MEX','MEXICO','MÉXICO') THEN 'MX'
            ELSE UPPER(TRIM(country))
        END AS country
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY merchant_id
                ORDER BY TRY_TO_TIMESTAMP(updated_at) DESC
            ) AS rn
        FROM kipu_pay.bronze.merchants
    )
    WHERE rn = 1
),

-- Cotizaciones normalizadas
fx AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(units_per_usd AS DECIMAL(18,6)) AS units_per_usd
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
),

-- Asignar última cotización disponible anterior o igual al pago
conversiones AS (
    SELECT
        t.transaction_id,
        t.merchant_id,
        t.currency,
        t.amount,
        c.merchant_name,
        c.country,
        f.units_per_usd,
        ROW_NUMBER() OVER (
            PARTITION BY t.transaction_id
            ORDER BY f.rate_date DESC
        ) AS rn
    FROM transacciones t
    LEFT JOIN comercios c
        ON t.merchant_id = c.merchant_id
    LEFT JOIN fx f
        ON t.currency = f.currency
       AND f.rate_date <= TO_DATE(t.fecha)
),

tpv_comercio AS (
    SELECT
        country,
        merchant_id,
        merchant_name,
        ROUND(SUM(
            CASE
                WHEN currency = 'USD' THEN amount
                ELSE amount / NULLIF(units_per_usd, 0)
            END
        ), 2) AS tpv_usd
    FROM conversiones
    WHERE rn = 1
      AND country IS NOT NULL
      AND (currency = 'USD' OR units_per_usd IS NOT NULL)
    GROUP BY country, merchant_id, merchant_name
),

ranking AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY tpv_usd DESC
        ) AS posicion
    FROM tpv_comercio
)

SELECT
    country,
    posicion,
    merchant_id,
    merchant_name,
    tpv_usd
FROM ranking
WHERE posicion <= 10
ORDER BY country, posicion;

In [0]:
%sql
SELECT
    source,
    UPPER(TRIM(currency)) AS currency,
    COUNT(*) AS registros,
    MIN(TRY_CAST(rate_date AS DATE)) AS primera_fecha,
    MAX(TRY_CAST(rate_date AS DATE)) AS ultima_fecha
FROM kipu_pay.bronze.fx_rates
GROUP BY source, UPPER(TRIM(currency))
ORDER BY source, currency;

In [0]:
%sql
WITH merchants_actuales AS (
    SELECT
        merchant_id,
        merchant_name,
        CASE
            WHEN UPPER(TRIM(country)) IN ('AR', 'ARG', 'ARGENTINA') THEN 'AR'
            WHEN UPPER(TRIM(country)) IN ('CL', 'CHL', 'CHILE') THEN 'CL'
            WHEN UPPER(TRIM(country)) IN ('CO', 'COL', 'COLOMBIA') THEN 'CO'
            WHEN UPPER(TRIM(country)) IN ('MX', 'MEX', 'MEXICO', 'MÉXICO') THEN 'MX'
            ELSE UPPER(TRIM(country))
        END AS country,
        ROW_NUMBER() OVER (
            PARTITION BY merchant_id
            ORDER BY TRY_TO_TIMESTAMP(updated_at) DESC
        ) AS rn
    FROM kipu_pay.bronze.merchants
),

transacciones AS (
    SELECT
        transaction_id,
        merchant_id,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(REPLACE(amount, ',', '.') AS DECIMAL(18,2)) AS amount,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
    WHERE LOWER(TRIM(status)) = 'approved'
),

fx AS (
    SELECT
        TRY_CAST(rate_date AS DATE) AS rate_date,
        UPPER(TRIM(currency)) AS currency,
        TRY_CAST(units_per_usd AS DECIMAL(18,6)) AS units_per_usd
    FROM kipu_pay.bronze.fx_rates
    WHERE TRY_CAST(units_per_usd AS DECIMAL(18,6)) > 0
),

transacciones_fx AS (
    SELECT
        t.transaction_id,
        t.merchant_id,
        t.currency,
        t.amount,
        CASE
            WHEN t.currency = 'USD' THEN t.amount
            ELSE t.amount / NULLIF(f.units_per_usd, 0)
        END AS amount_usd,
        ROW_NUMBER() OVER (
            PARTITION BY t.transaction_id
            ORDER BY f.rate_date DESC
        ) AS rn
    FROM transacciones t
    LEFT JOIN fx f
        ON t.currency = f.currency
       AND f.rate_date <= TO_DATE(t.fecha)
    WHERE t.fecha >= TIMESTAMP '2026-10-01 00:00:00'
      AND t.fecha < TIMESTAMP '2026-11-01 00:00:00'
      AND t.fecha IS NOT NULL
),

tpv_comercios AS (
    SELECT
        m.merchant_id,
        m.merchant_name,
        m.country,
        SUM(t.amount_usd) AS tpv_usd
    FROM transacciones_fx t
    INNER JOIN merchants_actuales m
        ON t.merchant_id = m.merchant_id
       AND m.rn = 1
    WHERE t.rn = 1
      AND t.amount_usd IS NOT NULL
    GROUP BY
        m.merchant_id,
        m.merchant_name,
        m.country
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY tpv_usd DESC
        ) AS posicion
    FROM tpv_comercios
)

SELECT
    country,
    posicion,
    merchant_id,
    merchant_name,
    ROUND(tpv_usd, 2) AS tpv_usd
FROM ranking
WHERE posicion <= 10
ORDER BY country, posicion;

In [0]:
%sql

WITH transacciones AS (
    SELECT
        created_at,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
)

SELECT
    COUNT(*) AS total,
    COUNT(fecha) AS fechas_convertidas,
    COUNT(*) - COUNT(fecha) AS fechas_invalidas
FROM transacciones;

In [0]:
%sql

WITH transacciones AS (
    SELECT
        *,
        COALESCE(
            TRY_TO_TIMESTAMP(created_at, 'dd/MM/yyyy HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, 'yyyy-MM-dd HH:mm:ss'),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ssXXX"),
            TRY_TO_TIMESTAMP(created_at, "yyyy-MM-dd'T'HH:mm:ss.SSSX")
        ) AS fecha
    FROM kipu_pay.bronze.transactions
)

SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN fecha >= TIMESTAMP '2026-10-01 00:00:00'
              AND fecha < TIMESTAMP '2026-11-01 00:00:00'
             THEN 1 ELSE 0 END) AS octubre,
    SUM(CASE WHEN currency = 'USD' THEN 1 ELSE 0 END) AS usd,
    SUM(CASE WHEN status = 'approved' THEN 1 ELSE 0 END) AS approved,
    SUM(CASE WHEN fecha >= TIMESTAMP '2026-10-01 00:00:00'
              AND fecha < TIMESTAMP '2026-11-01 00:00:00'
              AND currency = 'USD'
             THEN 1 ELSE 0 END) AS octubre_usd,
    SUM(CASE WHEN fecha >= TIMESTAMP '2026-10-01 00:00:00'
              AND fecha < TIMESTAMP '2026-11-01 00:00:00'
              AND status = 'approved'
             THEN 1 ELSE 0 END) AS octubre_approved
FROM transacciones;

In [0]:
%sql
SELECT
    merchant_id,
    COUNT(*) AS total
FROM kipu_pay.bronze.merchants
GROUP BY merchant_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM kipu_pay.bronze.merchants
WHERE merchant_id IN (
    SELECT merchant_id
    FROM kipu_pay.bronze.merchants
    GROUP BY merchant_id
    HAVING COUNT(*) > 1
)
ORDER BY merchant_id;